In [0]:
import os
from pathlib import Path
import csv
from loguru import logger
import configparser

In [0]:
# Install the loguru package
%pip install loguru

In [0]:
with open('dummy_data.csv', 'r') as f:
    reader = csv.reader(f)
    rows = list(reader)
    if len(rows) <= 1:
        print('dummy_data.csv is empty')
    else:
        print('dummy_data.csv is not empty')
        # Extract the 'name' column (index 1), skipping header
        names = [row[1] for row in rows[1:] if len(row) > 1]
        unique_names = list(set(names))
        print('Unique names:', unique_names)

In [0]:
#Use of config.ini file
config_file = Path('config.ini')
if config_file.exists():
    print('config.ini file exists')
else:
    print('config.ini file does not exist')

config = configparser.ConfigParser()
config.read('config.ini')

#reading database section from config file
database_name = config.get('database', 'database')
user = config.get('database', 'user')
password = config.get('database', 'password')
host = config.get('database', 'host')
port = config.get('database', 'port')

mysql_command = f"mysql -u {user} -p{password} -h {host} -P {port} {database_name}"
print(f'MySQL command: {mysql_command}')


# DBTITLE 1
#Use of loguru
logger.info('This is an info message')
logger.warning('This is a warning message')
logger.error('This is an error message')

# DBTITLE 1
#Use of os module
os.getcwd()
os.listdir('.')

#Remember : the data in ini file is always string format so we need to convert it to int or float if required
port = config.get('database', 'port')
print(f'Port datatype: {type(port)}')


In [0]:
%pip install pycryptodomex

In [0]:
import base64
from Cryptodome.Cipher import AES
from Cryptodome.Protocol.KDF import PBKDF2
import os, sys
from loguru import logger

try:
    key = "key_for_encryption"
    iv = "my_username_pass"
    salt = "salt"

    if not (key and iv and salt):
        raise Exception(F"Error while fetching details for key/iv/salt")
except Exception as e:
    logger.error("Error occurred. Details: %s", e)
    sys.exit(0)

BS = 16
pad = lambda s: bytes(s + (BS - len(s) % BS) * chr(BS - len(s) % BS), 'utf-8')
unpad = lambda s: s[0:-ord(s[-1:])]

def get_private_key():
    Salt = salt.encode('utf-8')
    kdf = PBKDF2(key, Salt, 64, 1000)
    key32 = kdf[:32]
    return key32


def encrypt(raw):
    raw = pad(raw)
    cipher = AES.new(get_private_key(), AES.MODE_CBC, iv.encode('utf-8'))
    return base64.b64encode(cipher.encrypt(raw))


def decrypt(enc):
    cipher = AES.new(get_private_key(), AES.MODE_CBC, iv.encode('utf-8'))
    return unpad(cipher.decrypt(base64.b64decode(enc))).decode('utf8')

print(encrypt("Tejas"))
print(decrypt("tkTXVsjQolTOn4fWN8le0A=="))
